# Train YOLO26s on the football dataset
Kitman's football default hyperparameters, written out in full, `imgsz=1920`, `batch=8`. Everything Ultralytics logs is also written to `<run dir>/train.log`.

In [ ]:
import logging
from datetime import datetime
from pathlib import Path

from ultralytics import YOLO

ROOT = Path(r"D:/College/DAU/Sem 03/Computer Vision/Project")
DATA = ROOT / "dataset" / "frames" / "data.yaml"
PROJECT = ROOT / "runs"
NAME = f"yolo26s_1920_b8_{datetime.now():%Y%m%d_%H%M%S}"

run_dir = PROJECT / NAME
run_dir.mkdir(parents=True, exist_ok=True)
log_path = run_dir / "train.log"

handler = logging.FileHandler(log_path, encoding="utf-8")
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logging.getLogger("ultralytics").addHandler(handler)
print("Logging to", log_path)

In [ ]:
model = YOLO("yolo26s.pt")  # downloaded by Ultralytics on first use
results = model.train(
    data=str(DATA), project=str(PROJECT), name=NAME, exist_ok=True,
    # basic
    imgsz=1920, epochs=100, batch=8, patience=25, seed=42,
    # optimiser and learning rate
    optimizer="AdamW", lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=3.0, cos_lr=False,
    # augmentation (football defaults: scale 0.3, hsv_s 0.5; Ultralytics uses 0.5, 0.7)
    fliplr=0.5, flipud=0.0, degrees=0.0, shear=0.0, perspective=0.0,
    scale=0.3, translate=0.1, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    mosaic=1.0, close_mosaic=10, mixup=0.0, cutmix=0.0, copy_paste=0.0, erasing=0.4,
    # run
    device=0, workers=2, cache=False, amp=True, freeze=None, rect=False,
    dropout=0.0, multi_scale=0.0, save_period=-1,
)

In [ ]:
# Final scores of best.pt on the test split
best = YOLO(run_dir / "weights" / "best.pt")
best.val(data=str(DATA), split="test", imgsz=1920, batch=8, project=str(PROJECT), name=f"{NAME}_test")

logging.getLogger("ultralytics").removeHandler(handler)
handler.close()